Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas
DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your
own roll number digits as follows:
• Take the LAST TWO DIGITS of your roll number. For each digit d, compute category = ["billing", "account",
"general"][d % 3]. Invent one realistic question+answer+3 keywords per entry that fits the assigned category (e.g.
if d%3 gives "account", write a question like “how do I update my registered mobile number”).
• # Example roll number ...23 -> digits 2, 3
• # digit 2 -> category[2 % 3] = general
• # digit 3 -> category[3 % 3] = billing

In [2]:
import pandas as pd

# Define your roll number here
roll_number = "1024170435"
last_two_digits = [int(roll_number[-2]), int(roll_number[-1])]

categories_map = ["billing", "account", "general"]

# Digit calculations
d1, d2 = last_two_digits[0], last_two_digits[1]
cat1 = categories_map[d1 % 3]
cat2 = categories_map[d2 % 3]

print(f"Roll Number: {roll_number}")
print(f"Digit {d1} -> {d1} % 3 = {d1 % 3} -> Category: '{cat1}'")
print(f"Digit {d2} -> {d2} % 3 = {d2 % 3} -> Category: '{cat2}'\n")

# 4 Fixed Entries provided in the PDF
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Personalized entries tailored for ["general", "billing"]
# (Adjust if your roll number yields different categories)
personalized_entries = [
    {
        "question": "where is your office located",
        "answer": "Our headquarters is located at Sector 62, City Center.",
        "keywords": "location address branch office",
        "category": cat1
    },
    {
        "question": "how to request a refund for failed transaction",
        "answer": "Go to Billing History and click 'Request Refund'.",
        "keywords": "refund invoice money transaction",
        "category": cat2
    }
]

# Combine and create the 6-row DataFrame
faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print("Final 6-row FAQ DataFrame:")
faq_df

Roll Number: 1024170435
Digit 3 -> 3 % 3 = 0 -> Category: 'billing'
Digit 5 -> 5 % 3 = 2 -> Category: 'general'

Final 6-row FAQ DataFrame:


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,where is your office located,"Our headquarters is located at Sector 62, City...",location address branch office,billing
5,how to request a refund for failed transaction,Go to Billing History and click 'Request Refund'.,refund invoice money transaction,general



Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching
entries ranked by confidence

In [ ]:
def score_hypothesis(query, df):
    """
    Computes confidence score based on keyword overlaps and question token matches.
    Returns matching entries ranked by confidence score in descending order.
    """
    query_tokens = set(query.lower().split())
    scored_results = []

    for idx, row in df.iterrows():
        kw_tokens = set(row["keywords"].lower().split())
        q_tokens = set(row["question"].lower().split())

        # Keyword matches carry 2 points weight, question matches carry 1 point weight
        kw_matches = len(query_tokens.intersection(kw_tokens))
        q_matches = len(query_tokens.intersection(q_tokens))

        score = (kw_matches * 2) + q_matches

        if score > 0:
            scored_results.append({
                "index": idx,
                "score": score,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"]
            })

    results_df = pd.DataFrame(scored_results)
    if not results_df.empty:
        results_df = results_df.sort_values(by="score", ascending=False).reset_index(drop=True)
    return results_df

# Demonstration
sample_query = "how to pay fee using upi"
print(f"Scoring results for query: '{sample_query}'")
score_hypothesis(sample_query, faq_df)

Scoring results for query: 'how to pay fee using upi'


,index,score,question,answer,category
0,3,9,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing
1,0,3,what is the annual fee,The annual fee is Rs 500.,billing
2,1,2,how to reset password,Go to Settings > Reset Password.,account
3,5,2,how to request a refund for failed transaction,Go to Billing History and click 'Request Refund'.,billing


Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it
using the category of one of the personalized entries from Q1, and print the result.

In [ ]:
def same_category(category_name, df):
    """
    Filters and returns all questions belonging to a specified category.
    """
    filtered_df = df[df["category"].str.lower() == category_name.lower()]
    return filtered_df[["question", "category"]].reset_index(drop=True)

# Calling with the category of the first personalized entry from Q1
target_category = cat1  # e.g., 'general'
print(f"Questions belonging to category '{target_category}':\n")
same_cat_results = same_category(target_category, faq_df)
same_cat_results

Questions belonging to category 'general':



,question,category
0,what are your working hours,general
1,where is your office located,general


Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and
save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [ ]:
# Pick entry at index 1 ('how to reset password')
selected_index = 1
print(f"Selected Question: {faq_df.loc[selected_index, 'question']}")
print(f"Original Keywords: {faq_df.loc[selected_index, 'keywords']}")

# Accept input from user (or fallback value if automated)
new_keyword = input("Enter a new keyword to append: ").strip()
if not new_keyword:
    new_keyword = "security"

# Append the new keyword
faq_df.loc[selected_index, "keywords"] += f" {new_keyword}"
print(f"\nUpdated Keywords: {faq_df.loc[selected_index, 'keywords']}")

# Save to CSV
csv_filename = f"{roll_number}_faq_data.csv"
faq_df.to_csv(csv_filename, index=False)
print(f"\nKnowledge base saved successfully to '{csv_filename}'.")

Selected Question: how to reset password
Original Keywords: password reset login

Updated Keywords: password reset login 1024170429

Knowledge base saved successfully to '1024170429_faq_data.csv'.


Q5: Using groupby, print how many FAQ entries you have per category

In [ ]:
# Group by category and compute counts
faq_distribution = faq_df.groupby("category").size().reset_index(name="count")
print("FAQ Entries per Category:")
faq_distribution

FAQ Entries per Category:


,category,count
0,account,1
1,billing,3
2,general,2


Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one
— it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that
produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [ ]:
def score_hypothesis_with_tie_handling(query, df):
    """
    Ranks matching entries by confidence score.
    Detects if there is a tie for the top score and displays all tied entries.
    """
    query_tokens = set(query.lower().split())
    scored_results = []

    for idx, row in df.iterrows():
        kw_tokens = set(row["keywords"].lower().split())
        q_tokens = set(row["question"].lower().split())

        # Scoring logic
        score = (len(query_tokens.intersection(kw_tokens)) * 2) + len(query_tokens.intersection(q_tokens))

        if score > 0:
            scored_results.append({
                "score": score,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"]
            })

    if not scored_results:
        print(f"Query: '{query}' -> No matches found.\n")
        return None

    results_df = pd.DataFrame(scored_results).sort_values(by="score", ascending=False).reset_index(drop=True)
    max_score = results_df["score"].max()
    top_matches = results_df[results_df["score"] == max_score]

    print(f"==================================================")
    print(f"Query: '{query}'")
    print(f"Highest Score: {max_score}")

    if len(top_matches) > 1:
        print(f"STATUS: TIE DETECTED ({len(top_matches)} entries tied for 1st place)")
    else:
        print("STATUS: UNIQUE BEST MATCH FOUND")
    print(f"==================================================")

    return top_matches

# Demonstration 1: Query that causes a tie between the two 'fee' entries
print("--- DEMO 1: QUERY WITH A TIE ---")
tie_query = "fee"
display(score_hypothesis_with_tie_handling(tie_query, faq_df))

print("\n--- DEMO 2: QUERY WITHOUT A TIE ---")
# Demonstration 2: Query with a unique best match
unique_query = "how to reset password"
display(score_hypothesis_with_tie_handling(unique_query, faq_df))

--- DEMO 1: QUERY WITH A TIE ---
Query: 'fee'
Highest Score: 3
STATUS: TIE DETECTED (2 entries tied for 1st place)


,score,question,answer,category
0,3,what is the annual fee,The annual fee is Rs 500.,billing
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing



--- DEMO 2: QUERY WITHOUT A TIE ---
Query: 'how to reset password'
Highest Score: 8
STATUS: UNIQUE BEST MATCH FOUND


,score,question,answer,category
0,8,how to reset password,Go to Settings > Reset Password.,account
